<a href="https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
!git clone https://github.com/Khuld13/ML-intern-starter.git
%cd ML-intern-starter
!pwd
!ls

fatal: destination path 'ML-intern-starter' already exists and is not an empty directory.
/content/ML-intern-starter/ML-intern-starter
/content/ML-intern-starter/ML-intern-starter
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [30]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

Connected.


In [31]:
import os

# Clone only if not already present (safe to re-run without erroring)
if not os.path.exists('/content/ML-intern-starter'):
    !git clone https://github.com/Khuld13/ML-intern-starter.git

%cd /content/ML-intern-starter
!pwd

# --- HF connection ---
from google.colab import userdata
import duckdb
import pandas as pd

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

# --- Views: dev partitions only (Feb + March), never the full table or _sample ---
con.sql("""
    CREATE OR REPLACE VIEW dim_clients AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""")
con.sql("""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""")
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")
con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")
print("Views ready.")

/content/ML-intern-starter
/content/ML-intern-starter
Connected.
Views ready.


In [32]:
con.sql("DESCRIBE SELECT * FROM dim_content").show()
con.sql("DESCRIBE SELECT * FROM fact_march").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [33]:
import pandas as pd
pd.set_option('display.max_rows', None)

print("=== dim_content ===")
print(con.sql("DESCRIBE SELECT * FROM dim_content").df().to_string())

print("\n=== fact_march ===")
print(con.sql("DESCRIBE SELECT * FROM fact_march").df().to_string())

=== dim_content ===
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         c

In [34]:
con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")

# Aggregate to one row per content item per month, then compare
monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)
""")

print(monthly_compare.df().shape)
monthly_compare.df().head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(303572, 4)


,content_hash_id,impressions_feb,impressions_march,declined_flag
0,content_d0dff76c889de68f,0.0,181.0,0
1,content_67741cce996cfafa,0.0,46.0,0
2,content_2e6360ad20fd7107,0.0,899.0,0
3,content_ac8663da7484669a,0.0,34.0,0
4,content_65c50dfe9d87a585,0.0,3108.0,0
5,content_d49a012dcb924e31,0.0,329.0,0
6,content_614baf2af4330bd7,0.0,772.0,0
7,content_4dc944b7d0b65ecc,0.0,134.0,0
8,content_4a1ca0fa5c177e0c,18.0,14.0,1
9,content_225dc9235023be5f,123.0,488.0,0


In [35]:
feb_availability = con.sql("""
    SELECT
        content_hash_id,
        MAX(client_has_gsc) AS had_gsc_feb,
        MAX(gsc_data_available) AS gsc_available_feb,
        SUM(gsc_impressions) AS impressions_feb
    FROM fact_feb
    GROUP BY content_hash_id
""").df()

# How many of the "zero impressions in Feb" rows are actually "no GSC data" in disguise?
zero_feb = feb_availability[feb_availability['impressions_feb'] == 0]
print("Total content with 0 impressions in Feb:", len(zero_feb))
print("Of those, had_gsc_feb == False:", (zero_feb['had_gsc_feb'] == False).sum())
print("Of those, gsc_available_feb == False:", (zero_feb['gsc_available_feb'] == False).sum())

Total content with 0 impressions in Feb: 167987
Of those, had_gsc_feb == False: 0
Of those, gsc_available_feb == False: 167987


In [36]:
monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)   -- inner join: only content measured in BOTH months
""")

print(monthly_compare.df().shape)
monthly_compare.df().head(10)

(134238, 4)


,content_hash_id,impressions_feb,impressions_march,declined_flag
0,content_4a1ca0fa5c177e0c,18.0,14.0,1
1,content_225dc9235023be5f,123.0,488.0,0
2,content_cfad137c1b04251b,150.0,438.0,0
3,content_9e7c70abfbae371e,17.0,2436.0,0
4,content_c0fc3b40ce00a5d7,51.0,335.0,0
5,content_f414582a500a5cc0,259.0,1808.0,0
6,content_f4003f159d792025,389.0,728.0,0
7,content_8274c3bd4b89c757,14.0,20.0,0
8,content_205d4e68446047b4,22.0,11.0,1
9,content_2adf1ccd6ae7f9c8,31.0,75.0,0


In [37]:
# Reference date = last day actually observed in the March partition
ref_date = con.sql("SELECT MAX(report_date) AS d FROM fact_march").df()['d'][0]
print("Reference date:", ref_date)

staleness = con.sql(f"""
    SELECT
        content_hash_id,
        content_updated_date,
        DATE '{ref_date}' - content_updated_date AS days_since_update
    FROM dim_content
""").df()

print(staleness.shape)
staleness.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reference date: 2026-03-31 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(519606, 3)


,content_hash_id,content_updated_date,days_since_update
0,content_004de9653278b5a4,2026-07-01,-92
1,content_00dc5efae381b2ab,2026-07-01,-92
2,content_01410f2556c327ac,2026-07-01,-92
3,content_019f27f634053ca7,2026-06-15,-76
4,content_01efa71faea45dcc,2026-06-01,-62
5,content_01fc9e2e57898b55,2026-07-01,-92
6,content_0212158fa61c5fcb,2026-07-01,-92
7,content_023d807c7d922db1,2026-06-13,-74
8,content_026f7405cc253242,2026-07-01,-92
9,content_02c8b23ea5bdb275,2026-07-01,-92


In [38]:
staleness_valid = staleness[staleness['days_since_update'] >= 0].copy()

print("Total content items:", len(staleness))
print("Excluded (updated after March ref date — staleness undetermined):", (staleness['days_since_update'] < 0).sum())
print("Usable for staleness check:", len(staleness_valid))

Total content items: 519606
Excluded (updated after March ref date — staleness undetermined): 382739
Usable for staleness check: 136867


In [39]:
monthly_compare_df = monthly_compare.df()

combined = monthly_compare_df.merge(staleness, on='content_hash_id', how='left')

print("Total in monthly_compare:", len(combined))
print("Of those, staleness unusable (updated after March ref):", (combined['days_since_update'] < 0).sum())
print("Of those, staleness usable:", (combined['days_since_update'] >= 0).sum())

Total in monthly_compare: 134238
Of those, staleness unusable (updated after March ref): 107091
Of those, staleness usable: 27147


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

**Signals checked**

1. **Staleness** (`days_since_update`, from `dim_content.content_updated_date`) — the signal behind FlyRank's real refresh flags.
2. **Volume** (`gsc_impressions`) — filters whether a page has enough demand to be worth fixing.

**Decline outcome used for validation**

No pre-built decline label exists in the warehouse, so I built one: content items with lower total GSC impressions in March 2026 vs. February 2026 (both dev-partition months, `gsc_data_available = TRUE` only, to avoid treating missing tracking as a real zero).

**Signal 1 — Staleness: verdict = OPPOSITE**

| days_since_update | n | decline_rate |
|---|---|---|
| 0-30 | 642 | 53.7% |
| 31-90 | 25,471 | 36.2% |
| 91-180 | 902 | 27.7% |
| 180+ | 132 | 34.1% |

Decline rate is highest for the *freshest* content and falls as staleness increases — the reverse of the "stale pages decline more" hypothesis. Likely explanation (unverified): edits are often reactive, made after a page is already declining, not preventive. Note: this check only covers content untouched since March (~27K of ~134K rows) — content edited again after March couldn't be scored, a real scope limit, not a bug.

**Signal 2 — Volume: verdict = CONFIRMED (as a reliability filter, not a decline predictor)**

| impressions_march | n | decline_rate |
|---|---|---|
| 0-50 | 36,572 | 42.9% |
| 51-250 | 29,188 | 27.6% |
| 251-500 | 14,048 | 26.9% |
| 500+ | 54,430 | 21.8% |

Low-volume pages show inflated apparent decline rates — consistent with noise from small denominators, where a few impressions swinging month to month can flip the flag. This confirms volume is needed as a minimum-threshold filter before trusting any decline signal, matching the guide's framing.

**Rule, in plain words**

Since staleness didn't behave as hypothesized, the rule doesn't use "old = bad." Instead: **flag a page if it shows a real month-over-month impression decline AND has enough volume (≥250 impressions) that the decline is unlikely to be noise.**

**Reason code:** `declining_with_demand`
**Action label:** `review_for_refresh`

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: Staleness bucket check
staleness_check = combined[combined['days_since_update'] >= 0].copy()
staleness_check['staleness_bucket'] = pd.cut(
    staleness_check['days_since_update'],
    bins=[-1, 30, 90, 180, float('inf')],
    labels=['0-30 days', '31-90 days', '91-180 days', '180+ days']
)
print(staleness_check.groupby('staleness_bucket', observed=True).agg(
    n=('content_hash_id', 'count'), decline_rate=('declined_flag', 'mean')
).reset_index())

# Signal 2: Volume bucket check
volume_check = monthly_compare_df.copy()
volume_check['volume_bucket'] = pd.cut(
    volume_check['impressions_march'],
    bins=[-1, 50, 250, 500, float('inf')],
    labels=['0-50', '51-250', '251-500', '500+']
)
print(volume_check.groupby('volume_bucket', observed=True).agg(
    n=('content_hash_id', 'count'), decline_rate=('declined_flag', 'mean')
).reset_index())

  staleness_bucket      n  decline_rate
0        0-30 days    642      0.537383
1       31-90 days  25471      0.361548
2      91-180 days    902      0.277162
3        180+ days    132      0.340909
  volume_bucket      n  decline_rate
0          0-50  36572      0.429427
1        51-250  29188      0.276381
2       251-500  14048      0.268579
3          500+  54430      0.217748


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

Score = percent decline (Feb → March), but only for pages meeting the rule (real decline + ≥250 impressions in March to filter noise, per Signal 2). Pages that don't meet the rule score 0 and sort to the bottom.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

scored = monthly_compare_df.copy()

# Percent decline — magnitude, not just direction
scored['pct_decline'] = (scored['impressions_feb'] - scored['impressions_march']) / scored['impressions_feb'].replace(0, pd.NA)

# The rule: real decline AND enough volume to trust the signal (Signal 2 verdict)
scored['meets_rule'] = (scored['declined_flag'] == 1) & (scored['impressions_march'] >= 250)

# Score only applies to pages meeting the rule; everyone else scores 0 and ranks last
scored['baseline_action_score'] = scored['pct_decline'].where(scored['meets_rule'], 0)

scored['reason_code'] = scored['meets_rule'].map({True: 'declining_with_demand', False: 'none'})
scored['action'] = scored['meets_rule'].map({True: 'review_for_refresh', False: 'no_action'})

scored = scored.sort_values('baseline_action_score', ascending=False).reset_index(drop=True)
scored['rank'] = scored.index + 1

os.makedirs('work/outputs', exist_ok=True)
output_cols = ['rank', 'content_hash_id', 'impressions_feb', 'impressions_march',
               'pct_decline', 'baseline_action_score', 'reason_code', 'action']
scored[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print(scored[output_cols].head(10))
print("\nTotal flagged:", scored['meets_rule'].sum(), "of", len(scored))

   rank           content_hash_id  impressions_feb  impressions_march  \
0     1  content_2ac8c7995de53cd1          92128.0             1926.0   
1     2  content_d16bbebfbb3c8fda          15238.0              555.0   
2     3  content_cf1af462f6317c31          30632.0             1143.0   
3     4  content_29c1e9fb447ec17f           8583.0              332.0   
4     5  content_93adb1b4fab6b97a           7227.0              303.0   
5     6  content_0709f29e7f096e6d          51000.0             2288.0   
6     7  content_f173fa70b277cf04           7758.0              352.0   
7     8  content_73aa32cad383d8f9          14011.0              668.0   
8     9  content_b85c606ed4e64eb2          24834.0             1237.0   
9    10  content_04235ab203bbfd92          26829.0             1339.0   

   pct_decline  baseline_action_score            reason_code  \
0     0.979094               0.979094  declining_with_demand   
1     0.963578               0.963578  declining_with_demand   
2   

In [45]:
print(scored['pct_decline'].replace([float('inf'), float('-inf')], pd.NA).isna().sum())

0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_ids = scored.head(20)['content_hash_id'].tolist()
ids_str = ", ".join(f"'{i}'" for i in top20_ids)

context = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        content_type,
        word_count,
        content_updated_date,
        is_published
    FROM dim_content
    WHERE content_hash_id IN ({ids_str})
""").df()

top20_review = scored.head(20).merge(context, on='content_hash_id', how='left')
top20_review

,content_hash_id,impressions_feb,impressions_march,declined_flag,pct_decline,meets_rule,baseline_action_score,reason_code,action,rank,client_hash_id,content_type,word_count,content_updated_date,is_published
0,content_2ac8c7995de53cd1,92128.0,1926.0,1,0.979094,True,0.979094,declining_with_demand,review_for_refresh,1,client_23a62021009f63c4,keyword article,3526,2026-06-28,True
1,content_d16bbebfbb3c8fda,15238.0,555.0,1,0.963578,True,0.963578,declining_with_demand,review_for_refresh,2,client_73cda7b4e4f265ea,keyword article,<NA>,2026-05-18,True
2,content_cf1af462f6317c31,30632.0,1143.0,1,0.962686,True,0.962686,declining_with_demand,review_for_refresh,3,client_23a62021009f63c4,keyword article,6770,2026-02-25,True
3,content_29c1e9fb447ec17f,8583.0,332.0,1,0.961319,True,0.961319,declining_with_demand,review_for_refresh,4,client_73cda7b4e4f265ea,keyword article,<NA>,2026-05-18,True
4,content_93adb1b4fab6b97a,7227.0,303.0,1,0.958074,True,0.958074,declining_with_demand,review_for_refresh,5,client_08a6a72ff48e62c0,keyword article,2963,2026-06-10,True
5,content_0709f29e7f096e6d,51000.0,2288.0,1,0.955137,True,0.955137,declining_with_demand,review_for_refresh,6,client_73cda7b4e4f265ea,keyword article,<NA>,2026-02-25,True
6,content_f173fa70b277cf04,7758.0,352.0,1,0.954627,True,0.954627,declining_with_demand,review_for_refresh,7,client_23a62021009f63c4,keyword article,6031,2026-02-25,True
7,content_73aa32cad383d8f9,14011.0,668.0,1,0.952323,True,0.952323,declining_with_demand,review_for_refresh,8,client_23a62021009f63c4,keyword article,6877,2026-02-25,True
8,content_b85c606ed4e64eb2,24834.0,1237.0,1,0.950189,True,0.950189,declining_with_demand,review_for_refresh,9,client_23a62021009f63c4,keyword article,6863,2026-02-25,True
9,content_04235ab203bbfd92,26829.0,1339.0,1,0.950091,True,0.950091,declining_with_demand,review_for_refresh,10,client_23a62021009f63c4,keyword article,5602,2026-02-25,True


In [48]:
top20_review['client_hash_id'].value_counts()

,count
client_hash_id,
client_23a62021009f63c4,11
client_73cda7b4e4f265ea,6
client_08a6a72ff48e62c0,2
client_62f4a7e64f5e0096,1


In [49]:
top20_review.groupby('client_hash_id')['content_updated_date'].apply(lambda x: x.value_counts())

client_hash_id                     
client_08a6a72ff48e62c0  2026-06-10     2
client_23a62021009f63c4  2026-02-25    10
                         2026-06-28     1
client_62f4a7e64f5e0096  2026-07-03     1
client_73cda7b4e4f265ea  2026-05-18     4
                         2026-02-25     2
Name: content_updated_date, dtype: int64

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.